# 02 - Classical Models (Fast, Updated)

This notebook trains the 4 classical models for both tasks:

- Logistic Regression
- KNN
- SVM (**LinearSVC for speed**)
- Random Forest

It is designed to be **Colab-friendly and fast**:

- upload `taskA_dataset.csv` and `taskB_dataset.csv`
- keeps the **full test set**
- uses **sampling only for the slowest models**
- saves results and prediction CSVs

## Expected runtime
This version is much faster than the earlier one because it avoids the slow RBF SVM and reduces training size only where needed.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, brier_score_loss
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', 200)

## 1) Upload the datasets

When prompted, upload:

- `taskA_dataset.csv`
- `taskB_dataset.csv`

In [ ]:
# File setup
OUTPUT_DIR = '/content/ds4400_valorant/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

from google.colab import files
print("Please upload taskA_dataset.csv and taskB_dataset.csv")
uploaded = files.upload()

taskA = pd.read_csv('taskA_dataset.csv', parse_dates=['Date'])
taskB = pd.read_csv('taskB_dataset.csv', parse_dates=['Date'])

print("Task A shape:", taskA.shape)
print("Task B shape:", taskB.shape)
taskA.head(2)

## 2) Runtime settings

These settings are chosen to keep the notebook practical on Colab without changing the project story.

- **Task A** is the baseline, so it can use a smaller training sample
- **KNN** and **SVM** are the slowest models, so they use a training subset
- **All models are evaluated on the full test set**

In [ ]:
# Speed / reproducibility settings
RANDOM_STATE = 42
FAST_MODE = True

# Task A is only the baseline, so we can safely cap its training size
TASKA_MAX_TRAIN = 40000

# Slow models use a smaller training sample
SLOW_MODEL_MAX_TRAIN = 30000

# Random Forest size
RF_N_ESTIMATORS = 150
RF_MAX_DEPTH = 20
RF_MIN_SAMPLES_LEAF = 2

## 3) Helper functions

In [ ]:
def maybe_sample_train(X, y, max_rows, random_state=42):
    if len(X) <= max_rows:
        return X.copy(), y.copy()
    sampled_idx = y.sample(n=max_rows, random_state=random_state).index
    return X.loc[sampled_idx].copy(), y.loc[sampled_idx].copy()


def split_train_test(df, target_col='Team1 Win'):
    train_df = df[df['Split'] == 'train'].copy()
    test_df = df[df['Split'] == 'test'].copy()

    X_train = train_df.drop(columns=['Date', 'Split', target_col], errors='ignore')
    y_train = train_df[target_col].copy()

    X_test = test_df.drop(columns=['Date', 'Split', target_col], errors='ignore')
    y_test = test_df[target_col].copy()

    return X_train, X_test, y_train, y_test


def make_preprocessor(X):
    cat_cols = [c for c in X.columns if c == 'Map']
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]), cat_cols)
        ],
        remainder='drop'
    )

    return preprocessor


def build_models(X_train):
    pre = make_preprocessor(X_train)

    models = {
        'Logistic Regression': Pipeline([
            ('pre', pre),
            ('clf', LogisticRegression(max_iter=1500, random_state=RANDOM_STATE))
        ]),
        'KNN': Pipeline([
            ('pre', pre),
            ('clf', KNeighborsClassifier(n_neighbors=21, n_jobs=-1))
        ]),
        'SVM': Pipeline([
            ('pre', pre),
            ('clf', LinearSVC(C=1.0, random_state=RANDOM_STATE, dual=False, max_iter=5000))
        ]),
        'Random Forest': Pipeline([
            ('pre', pre),
            ('clf', RandomForestClassifier(
                n_estimators=RF_N_ESTIMATORS,
                max_depth=RF_MAX_DEPTH,
                min_samples_leaf=RF_MIN_SAMPLES_LEAF,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])
    }

    return models


def evaluate_model(name, model, X_train, X_test, y_train, y_test, task_name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    probas = None
    scores = None

    if hasattr(model, "predict_proba"):
        probas = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, probas)
        brier = brier_score_loss(y_test, probas)
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
        roc_auc = roc_auc_score(y_test, scores)
        brier = np.nan
    else:
        roc_auc = np.nan
        brier = np.nan

    results = {
        'Task': task_name,
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds, zero_division=0),
        'F1': f1_score(y_test, preds, zero_division=0),
        'ROC_AUC': roc_auc,
        'Brier': brier
    }

    pred_df = pd.DataFrame({
        'y_true': y_test.values,
        'y_pred': preds
    })

    if probas is not None:
        pred_df['y_prob'] = probas
    elif scores is not None:
        pred_df['y_score'] = scores

    return results, pred_df, model

## 4) Run Task A

Task A is just the same-map baseline, so in `FAST_MODE` we cap the training size.

In [ ]:
X_train_A, X_test_A, y_train_A, y_test_A = split_train_test(taskA)

if FAST_MODE:
    X_train_A_use, y_train_A_use = maybe_sample_train(
        X_train_A, y_train_A, TASKA_MAX_TRAIN, RANDOM_STATE
    )
else:
    X_train_A_use, y_train_A_use = X_train_A, y_train_A

print("Task A train used:", X_train_A_use.shape)
print("Task A test used :", X_test_A.shape)

models_A = build_models(X_train_A_use)

taskA_results = []
taskA_best_models = {}

for model_name, model in models_A.items():
    print(f'Running Task A - {model_name}')

    X_fit, y_fit = X_train_A_use, y_train_A_use

    if FAST_MODE and model_name in ['KNN', 'SVM']:
        X_fit, y_fit = maybe_sample_train(
            X_train_A_use, y_train_A_use, SLOW_MODEL_MAX_TRAIN, RANDOM_STATE
        )
        print(f'  sampled train shape for {model_name}:', X_fit.shape)

    results, pred_df, fitted_model = evaluate_model(
        model_name, model,
        X_fit, X_test_A, y_fit, y_test_A,
        task_name='Task A'
    )

    taskA_results.append(results)
    taskA_best_models[model_name] = fitted_model

    pred_path = os.path.join(OUTPUT_DIR, f"taskA_preds_{model_name.lower().replace(' ', '_')}.csv")
    pred_df.to_csv(pred_path, index=False)

taskA_results_df = pd.DataFrame(taskA_results).sort_values('F1', ascending=False)
taskA_results_df

## 5) Run Task B

Task B is the main project, so Logistic Regression and Random Forest use the full training set.  
The slow models (**KNN** and **SVM**) still use a smaller training sample in `FAST_MODE`.

In [ ]:
X_train_B, X_test_B, y_train_B, y_test_B = split_train_test(taskB)

print("Task B full train:", X_train_B.shape)
print("Task B full test :", X_test_B.shape)

models_B = build_models(X_train_B)

taskB_results = []
taskB_best_models = {}

for model_name, model in models_B.items():
    print(f'Running Task B - {model_name}')

    X_fit, y_fit = X_train_B, y_train_B

    if FAST_MODE and model_name in ['KNN', 'SVM']:
        X_fit, y_fit = maybe_sample_train(
            X_train_B, y_train_B, SLOW_MODEL_MAX_TRAIN, RANDOM_STATE
        )
        print(f'  sampled train shape for {model_name}:', X_fit.shape)

    results, pred_df, fitted_model = evaluate_model(
        model_name, model,
        X_fit, X_test_B, y_fit, y_test_B,
        task_name='Task B'
    )

    taskB_results.append(results)
    taskB_best_models[model_name] = fitted_model

    pred_path = os.path.join(OUTPUT_DIR, f"taskB_preds_{model_name.lower().replace(' ', '_')}.csv")
    pred_df.to_csv(pred_path, index=False)

taskB_results_df = pd.DataFrame(taskB_results).sort_values('F1', ascending=False)
taskB_results_df

## 6) Save results

In [ ]:
taskA_results_path = os.path.join(OUTPUT_DIR, 'taskA_classical_results.csv')
taskB_results_path = os.path.join(OUTPUT_DIR, 'taskB_classical_results.csv')

taskA_results_df.to_csv(taskA_results_path, index=False)
taskB_results_df.to_csv(taskB_results_path, index=False)

print("Saved:")
print(taskA_results_path)
print(taskB_results_path)

## 7) Quick comparison

In [ ]:
all_results = pd.concat([taskA_results_df, taskB_results_df], ignore_index=True)
all_results = all_results.sort_values(['Task', 'F1'], ascending=[True, False])

display(all_results)

print("Task A best model:")
display(taskA_results_df.iloc[[0]])

print("Task B best model:")
display(taskB_results_df.iloc[[0]])

## 8) Show saved output files

In [ ]:
for fname in sorted(os.listdir(OUTPUT_DIR)):
    print(fname)